<a href="https://colab.research.google.com/github/Keerthan-AC-583/BIS-Lab/blob/main/1BM24CS133_CS_EV_Charging_Placement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import math
import numpy as np

CANDIDATE_SITES = [
    {"name": "Site A", "x": 1.0, "y": 1.0, "install_cost": 900000, "annual_operating_cost": 90000, "capacity": 420},
    {"name": "Site B", "x": 2.0, "y": 6.0, "install_cost": 850000, "annual_operating_cost": 85000, "capacity": 380},
    {"name": "Site C", "x": 3.5, "y": 2.5, "install_cost": 950000, "annual_operating_cost": 95000, "capacity": 450},
    {"name": "Site D", "x": 4.0, "y": 7.5, "install_cost": 800000, "annual_operating_cost": 82000, "capacity": 400},
    {"name": "Site E", "x": 5.5, "y": 4.0, "install_cost": 780000, "annual_operating_cost": 80000, "capacity": 360},
    {"name": "Site F", "x": 6.5, "y": 1.5, "install_cost": 880000, "annual_operating_cost": 88000, "capacity": 410},
    {"name": "Site G", "x": 7.5, "y": 6.5, "install_cost": 820000, "annual_operating_cost": 83000, "capacity": 390},
    {"name": "Site H", "x": 8.5, "y": 3.0, "install_cost": 920000, "annual_operating_cost": 92000, "capacity": 430},
    {"name": "Site I", "x": 9.0, "y": 8.0, "install_cost": 760000, "annual_operating_cost": 78000, "capacity": 350},
    {"name": "Site J", "x": 10.0, "y": 5.0, "install_cost": 870000, "annual_operating_cost": 87000, "capacity": 420},
    {"name": "Site K", "x": 1.5, "y": 9.0, "install_cost": 810000, "annual_operating_cost": 81000, "capacity": 390},
    {"name": "Site L", "x": 6.0, "y": 9.0, "install_cost": 840000, "annual_operating_cost": 84000, "capacity": 400},
]

DEMAND_POINTS = [
    {"name": "Demand 1", "x": 0.8, "y": 1.8, "demand": 85},
    {"name": "Demand 2", "x": 1.8, "y": 4.8, "demand": 70},
    {"name": "Demand 3", "x": 2.8, "y": 2.0, "demand": 95},
    {"name": "Demand 4", "x": 3.2, "y": 6.8, "demand": 75},
    {"name": "Demand 5", "x": 4.8, "y": 3.8, "demand": 110},
    {"name": "Demand 6", "x": 5.8, "y": 1.0, "demand": 80},
    {"name": "Demand 7", "x": 6.5, "y": 5.2, "demand": 105},
    {"name": "Demand 8", "x": 7.8, "y": 7.3, "demand": 90},
    {"name": "Demand 9", "x": 8.8, "y": 2.2, "demand": 100},
    {"name": "Demand 10", "x": 9.5, "y": 5.5, "demand": 95},
    {"name": "Demand 11", "x": 2.0, "y": 8.5, "demand": 65},
    {"name": "Demand 12", "x": 5.7, "y": 8.2, "demand": 80},
    {"name": "Demand 13", "x": 10.2, "y": 8.5, "demand": 60},
    {"name": "Demand 14", "x": 0.5, "y": 6.0, "demand": 70},
    {"name": "Demand 15", "x": 7.0, "y": 0.8, "demand": 85},
]

NUMBER_OF_STATIONS = 4
NUMBER_OF_NESTS = 30
MAX_ITERATIONS = 250
DISCOVERY_PROBABILITY = 0.25
STEP_SIZE = 0.80
MAX_SERVICE_DISTANCE_KM = 4.0
RANDOM_SEED = 42

WEIGHTS = {
    "installation": 0.18,
    "operating": 0.12,
    "travel": 0.20,
    "uncovered": 0.25,
    "overload": 0.25,
}

SITE_COORDS = np.array([[s["x"], s["y"]] for s in CANDIDATE_SITES], dtype=float)
SITE_INSTALL_COST = np.array([s["install_cost"] for s in CANDIDATE_SITES], dtype=float)
SITE_OPERATING_COST = np.array([s["annual_operating_cost"] for s in CANDIDATE_SITES], dtype=float)
SITE_CAPACITY = np.array([s["capacity"] for s in CANDIDATE_SITES], dtype=float)

DEMAND_COORDS = np.array([[d["x"], d["y"]] for d in DEMAND_POINTS], dtype=float)
DEMAND_VALUES = np.array([d["demand"] for d in DEMAND_POINTS], dtype=float)


def repair_solution(position):
    number_of_sites = len(CANDIDATE_SITES)
    clipped_position = np.clip(position, 0, number_of_sites - 1)
    selected_indexes = []

    for value in clipped_position:
        nearest_indexes = np.argsort(np.abs(np.arange(number_of_sites) - value))
        for index in nearest_indexes:
            index = int(index)
            if index not in selected_indexes:
                selected_indexes.append(index)
                break

    return np.array(selected_indexes, dtype=int)


def evaluate_solution(position):
    selected_indexes = repair_solution(position)
    selected_coords = SITE_COORDS[selected_indexes]

    distance_matrix = np.linalg.norm(
        DEMAND_COORDS[:, np.newaxis, :] - selected_coords[np.newaxis, :, :],
        axis=2,
    )

    nearest_station = np.argmin(distance_matrix, axis=1)
    nearest_distance = distance_matrix[np.arange(len(DEMAND_POINTS)), nearest_station]

    station_load = np.bincount(
        nearest_station,
        weights=DEMAND_VALUES,
        minlength=NUMBER_OF_STATIONS,
    )
    selected_capacity = SITE_CAPACITY[selected_indexes]

    total_demand = float(np.sum(DEMAND_VALUES))
    average_distance = float(np.average(nearest_distance, weights=DEMAND_VALUES))
    uncovered_demand = float(
        np.sum(DEMAND_VALUES[nearest_distance > MAX_SERVICE_DISTANCE_KM])
    )
    overload = np.maximum(station_load - selected_capacity, 0.0)
    total_overload = float(np.sum(overload))

    installation_cost = float(np.sum(SITE_INSTALL_COST[selected_indexes]))
    operating_cost = float(np.sum(SITE_OPERATING_COST[selected_indexes]))

    installation_score = installation_cost / (
        NUMBER_OF_STATIONS * float(np.max(SITE_INSTALL_COST))
    )
    operating_score = operating_cost / (
        NUMBER_OF_STATIONS * float(np.max(SITE_OPERATING_COST))
    )
    travel_score = average_distance / MAX_SERVICE_DISTANCE_KM
    uncovered_score = uncovered_demand / total_demand
    overload_score = total_overload / total_demand

    fitness = (
        WEIGHTS["installation"] * installation_score
        + WEIGHTS["operating"] * operating_score
        + WEIGHTS["travel"] * travel_score
        + WEIGHTS["uncovered"] * uncovered_score
        + WEIGHTS["overload"] * overload_score
    )

    metrics = {
        "selected_indexes": selected_indexes,
        "installation_cost": installation_cost,
        "operating_cost": operating_cost,
        "average_distance": average_distance,
        "uncovered_demand": uncovered_demand,
        "uncovered_percentage": 100.0 * uncovered_demand / total_demand,
        "station_load": station_load,
        "station_capacity": selected_capacity,
        "total_overload": total_overload,
        "nearest_distance": nearest_distance,
        "nearest_station": nearest_station,
        "fitness": float(fitness),
    }

    return float(fitness), metrics


def levy_flight(size, beta=1.5):
    numerator = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    denominator = (
        math.gamma((1 + beta) / 2)
        * beta
        * (2 ** ((beta - 1) / 2))
    )
    sigma = (numerator / denominator) ** (1 / beta)

    u = np.random.normal(0, sigma, size=size)
    v = np.random.normal(0, 1, size=size)
    return u / (np.abs(v) ** (1 / beta) + 1e-12)


def cuckoo_search():
    number_of_sites = len(CANDIDATE_SITES)

    nests = np.array([
        np.random.choice(number_of_sites, NUMBER_OF_STATIONS, replace=False)
        for _ in range(NUMBER_OF_NESTS)
    ], dtype=float)

    fitness_values = np.array([evaluate_solution(nest)[0] for nest in nests])

    best_index = int(np.argmin(fitness_values))
    best_nest = nests[best_index].copy()
    best_fitness = float(fitness_values[best_index])

    number_to_abandon = max(
        1, int(round(DISCOVERY_PROBABILITY * NUMBER_OF_NESTS))
    )

    for iteration in range(1, MAX_ITERATIONS + 1):
        for i in range(NUMBER_OF_NESTS):
            step = levy_flight(NUMBER_OF_STATIONS)
            new_position = (
                nests[i] + STEP_SIZE * step * (nests[i] - best_nest)
            )
            new_nest = repair_solution(new_position).astype(float)
            new_fitness = evaluate_solution(new_nest)[0]

            if new_fitness < fitness_values[i]:
                nests[i] = new_nest
                fitness_values[i] = new_fitness

        worst_indexes = np.argsort(fitness_values)[-number_to_abandon:]
        for index in worst_indexes:
            random_nest = np.random.choice(
                number_of_sites, NUMBER_OF_STATIONS, replace=False
            ).astype(float)
            nests[index] = random_nest
            fitness_values[index] = evaluate_solution(random_nest)[0]

        current_best_index = int(np.argmin(fitness_values))
        if fitness_values[current_best_index] < best_fitness:
            best_fitness = float(fitness_values[current_best_index])
            best_nest = nests[current_best_index].copy()

        if iteration == 1 or iteration % 50 == 0:
            print(
                f"Iteration {iteration:3d}/{MAX_ITERATIONS} | "
                f"Best fitness: {best_fitness:.6f}"
            )

    return repair_solution(best_nest), best_fitness


def display_results(best_solution, best_fitness):
    _, metrics = evaluate_solution(best_solution)

    print("\n" + "=" * 65)
    print("OPTIMAL EV CHARGING STATION PLACEMENT USING CUCKOO SEARCH")
    print("=" * 65)
    print(f"Best fitness value             : {best_fitness:.6f}")
    print(f"Total installation cost        : INR {metrics['installation_cost']:,.2f}")
    print(f"Annual operating cost          : INR {metrics['operating_cost']:,.2f}")
    print(f"Demand-weighted average distance: {metrics['average_distance']:.2f} km")
    print(f"Uncovered demand                : {metrics['uncovered_demand']:.1f} units/day")
    print(f"Uncovered demand percentage     : {metrics['uncovered_percentage']:.2f}%")
    print(f"Total capacity overload         : {metrics['total_overload']:.1f} units/day")

    print("\nSELECTED CHARGING STATIONS")
    for local_index, site_index in enumerate(metrics["selected_indexes"]):
        site = CANDIDATE_SITES[int(site_index)]
        load = metrics["station_load"][local_index]
        capacity = metrics["station_capacity"][local_index]
        print(
            f"{site['name']} | Coordinates: ({site['x']}, {site['y']}) | "
            f"Assigned demand: {load:.1f} | Capacity: {capacity:.1f}"
        )

    print("\nDEMAND-TO-STATION ASSIGNMENTS")
    selected_indexes = metrics["selected_indexes"]
    for i, demand in enumerate(DEMAND_POINTS):
        local_station_index = int(metrics["nearest_station"][i])
        site_index = int(selected_indexes[local_station_index])
        station_name = CANDIDATE_SITES[site_index]["name"]
        distance = metrics["nearest_distance"][i]
        status = (
            "COVERED"
            if distance <= MAX_SERVICE_DISTANCE_KM
            else "OUTSIDE SERVICE RANGE"
        )
        print(
            f"{demand['name']:12s} -> {station_name:7s} | "
            f"Distance: {distance:.2f} km | "
            f"Demand: {demand['demand']} | {status}"
        )


def main():
    if not 1 <= NUMBER_OF_STATIONS <= len(CANDIDATE_SITES):
        raise ValueError("NUMBER_OF_STATIONS must be between 1 and the number of candidate sites.")
    if NUMBER_OF_NESTS < 2:
        raise ValueError("NUMBER_OF_NESTS must be at least 2.")
    if MAX_ITERATIONS < 1:
        raise ValueError("MAX_ITERATIONS must be at least 1.")
    if not 0 <= DISCOVERY_PROBABILITY <= 1:
        raise ValueError("DISCOVERY_PROBABILITY must be between 0 and 1.")
    if not math.isclose(sum(WEIGHTS.values()), 1.0, rel_tol=1e-9, abs_tol=1e-9):
        raise ValueError("The objective weights must add up to 1.0.")
    if np.sum(DEMAND_VALUES) <= 0:
        raise ValueError("Total EV demand must be positive.")

    np.random.seed(RANDOM_SEED)

    print("Starting Cuckoo Search for EV charging station placement...\n")
    best_solution, best_fitness = cuckoo_search()
    display_results(best_solution, best_fitness)


if __name__ == "__main__":
    main()



Starting Cuckoo Search for EV charging station placement...

Iteration   1/250 | Best fitness: 0.380384
Iteration  50/250 | Best fitness: 0.368393
Iteration 100/250 | Best fitness: 0.368393
Iteration 150/250 | Best fitness: 0.368393
Iteration 200/250 | Best fitness: 0.368393
Iteration 250/250 | Best fitness: 0.368393

OPTIMAL EV CHARGING STATION PLACEMENT USING CUCKOO SEARCH
Best fitness value             : 0.368393
Total installation cost        : INR 3,450,000.00
Annual operating cost          : INR 348,000.00
Demand-weighted average distance: 1.90 km
Uncovered demand                : 0.0 units/day
Uncovered demand percentage     : 0.00%
Total capacity overload         : 0.0 units/day

SELECTED CHARGING STATIONS
Site G | Coordinates: (7.5, 6.5) | Assigned demand: 350.0 | Capacity: 390.0
Site D | Coordinates: (4.0, 7.5) | Assigned demand: 290.0 | Capacity: 400.0
Site C | Coordinates: (3.5, 2.5) | Assigned demand: 360.0 | Capacity: 450.0
Site F | Coordinates: (6.5, 1.5) | Assigned dema